# Experiments — Geographic inference mask

Builds Pantanal overlap mask for Kaggle inference. No GPU training.


In [1]:
from google.colab import drive
drive.mount("/content/drive")
%run colab_init.py

!pip install -q pandas matplotlib seaborn kaggle

from birdclef.colab import mount_and_configure, set_kaggle_token

mount_and_configure()
set_kaggle_token()



Mounted at /content/drive
Working directory: /content/drive/MyDrive/BirdCLEF_Project/repro
Copied train.csv to /content/train.csv
Copied sample_submission.csv to /content/sample_submission.csv


In [2]:
import numpy as np
import pandas as pd
from birdclef.paths import METADATA_DIR



In [4]:
import pandas as pd

from birdclef.paths import METADATA_DIR

full_df = pd.read_csv(METADATA_DIR / "train.csv")
target_columns = sorted(full_df["primary_label"].unique().tolist())
num_classes = len(target_columns)

bird_geo_bounds = full_df.groupby('primary_label').agg({
    'latitude': ['min', 'max'],
    'longitude': ['min', 'max']
})
bird_bounds_dict = bird_geo_bounds.to_dict(orient="index")

TEST_LAT_MIN, TEST_LAT_MAX = -21.6, -16.5
TEST_LON_MIN, TEST_LON_MAX = -57.6, -55.9
TOLERANCE = 5.0

pantanal_mask = np.zeros(num_classes, dtype=np.float32)
for i, bird in enumerate(target_columns):
    if bird in bird_bounds_dict:
        bounds = bird_bounds_dict[bird]
        b_lat_min = bounds[("latitude", "min")] - TOLERANCE
        b_lat_max = bounds[("latitude", "max")] + TOLERANCE
        b_lon_min = bounds[("longitude", "min")] - TOLERANCE
        b_lon_max = bounds[("longitude", "max")] + TOLERANCE
        lat_overlap = (b_lat_min <= TEST_LAT_MAX) and (b_lat_max >= TEST_LAT_MIN)
        lon_overlap = (b_lon_min <= TEST_LON_MAX) and (b_lon_max >= TEST_LON_MIN)
        if lat_overlap and lon_overlap:
            pantanal_mask[i] = 1.0
    else:
        pantanal_mask[i] = 1.0

allowed = int(np.sum(pantanal_mask))
print(f"Allowed birds: {allowed} | Masked out: {num_classes - allowed}")
final_kaggle_mask = np.where(pantanal_mask == 0.0, 0.01, 1.0)
print("PANTANAL_MASK = np.array(" + repr(final_kaggle_mask.tolist()) + ", dtype=np.float32)")



Allowed birds: 192 | Masked out: 14
PANTANAL_MASK = np.array([1.0, 0.01, 1.0, 0.01, 0.01, 1.0, 1.0, 1.0, 1.0, 1.0, 0.01, 0.01, 0.01, 0.01, 1.0, 0.01, 0.01, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.01, 1.0, 1.0, 1.0, 1.0, 0.01, 1.0, 0.01, 1.0, 1.0, 1.0, 1.0, 1.0, 0.01, 1.0, 1.0, 1.0, 1.0, 0.01, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,